# デートBot Phase A — Gemini一本モード（Colab）

要件: 複数ターンで条件ヒアリング → **Gemini 1回で slots更新＋案3つ＋理由**

LoRA版ノート（`date_bot_phase_a_*.ipynb`）の代替。GPU不要。CPUでも動く。

**事前準備**
1. ランタイムは **CPU でOK**（GPU不要）
2. 左の鍵アイコン Secrets に `GEMINI_API_KEY` を登録（Google AI Studio の無料キー）
3. 任意で `date_bot_preferences_summary.md` もアップロード

店舗CSVは使いません。エリア＋好み要約のみです。

**旧ノートとの対応**
| 旧（LoRA版） | 本ノート |
|---|---|
| 0 ライブラリ | 0（軽量） |
| 1 設定・好み | 1（同じ） |
| 2〜5 LoRA学習 | **スキップ**（説明のみ） |
| 6 Gemini | 6（同じ） |
| 7 対話エンジン | 7（Gemini一本） |
| 8 対話テスト | 8（同じ操作感） |
| 9 運用 | 9（本モード向け） |


## 0. ライブラリインストール


In [ ]:
# Gemini一本モード: ローカルLLM不要。requests のみあれば動く。
# Colab 標準の requests で足りるが、念のため入れる。
!pip -q install -U "requests"
print("install done")

## 1. 設定・好み要約の読み込み

LoRA版と同じ定数・好み要約。`torch` / `MODEL_ID` は使わない。


In [ ]:
import os, re, json
from pathlib import Path

DEFAULT_BUDGET = 3000
PRIORITY_AREAS = ["渋谷", "上野", "新宿", "お台場", "品川"]
MODE = "gemini_only"  # LoRA版と区別用

PREFERENCES_FALLBACK = (
    "呼称: リョウスケ君 / こゆたん\n"
    "エリア: 東京23区。優先は渋谷・上野・新宿／お台場〜品川\n"
    "予算デフォルト: 一人3000円\n"
    "好き: ご飯(おいしい+安い)、カフェ、映画、動物園・水族館、季節イベント、散歩・公園、スイーツ、猫カフェ、美術館・博物館\n"
    "NG: 高いのに微妙な店、長時間ダラダラ\n"
    "定番: イベント+ご飯、上野/渋谷起点、コスパ重視、微妙店のリベンジ\n"
    "口調: フランク。予約代行・在庫確認・恋愛深掘りはしない。個人情報は扱わない。"
)

prefs_path = Path("date_bot_preferences_summary.md")
if prefs_path.exists():
    PREFERENCES_TEXT = prefs_path.read_text(encoding="utf-8")
    print("Loaded preferences from file")
else:
    PREFERENCES_TEXT = PREFERENCES_FALLBACK
    print("Using embedded preferences fallback")

print("MODE:", MODE)
print(PREFERENCES_TEXT[:400], "...")

## 2〜5. LoRA関連はスキップ

本ノートではローカルモデル・学習CSV・トークナイズ・LoRA学習を行わない。

- GPU制限時の一時アーキテクチャ
- slots の状態は Python 側で保持し、各ターン Gemini に「これまでのslots＋今回発話」を渡す
- LoRA版に戻すときは旧ノート＋ Drive の `date_bot_lora` を使う（`docs/SAVE_LOAD.md`）


In [ ]:
# スキップ確認セル（実行してOK）
print("sections 2-5 skipped (no LoRA / no training)")
print("次はセクション6（Geminiセットアップ）へ")

## 6. Gemini セットアップ（無料枠）


In [ ]:
import os
import time
import requests

def get_gemini_api_key():
    if "GEMINI_API_KEY" in globals() and globals().get("GEMINI_API_KEY"):
        print("APIキー: 既存変数から取得")
        return str(globals()["GEMINI_API_KEY"]).strip()
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            print("APIキー: Colab Secrets から取得")
            return key.strip()
    except Exception as e:
        print(f"Colab Secrets 読み込み失敗: {e}")
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        print("APIキー: 環境変数から取得")
        return key.strip()
    return None

GEMINI_API_KEY = get_gemini_api_key()
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY がありません。")

# 確認済みモデル（LoRA版と同じ）
GEMINI_MODEL_NAME = "gemini-3.1-flash-lite"


def gemini_rest_generate(prompt: str, model_name: str = None, timeout: int = 45) -> str:
    model_name = model_name or GEMINI_MODEL_NAME
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent"
    r = requests.post(
        url,
        params={"key": GEMINI_API_KEY},
        json={"contents": [{"parts": [{"text": prompt}]}]},
        timeout=timeout,
    )
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:400]}")
    data = r.json()
    return data["candidates"][0]["content"]["parts"][0]["text"].strip()


sample = gemini_rest_generate("1語だけ返して", timeout=25)
print("Gemini REST ready:", GEMINI_MODEL_NAME, "|", sample)
gemini_model = GEMINI_MODEL_NAME

In [ ]:
# 診断用（任意）。絶対に変数名 model を使わないこと！
import requests
key = GEMINI_API_KEY
for cand_name in ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.5-flash"]:
    r = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{cand_name}:generateContent",
        params={"key": key},
        json={"contents": [{"parts": [{"text": "こんにちは。1語だけ返して。"}]}]},
        timeout=25,
    )
    print(cand_name, r.status_code, r.text[:150].replace(chr(10), " "))


## 7. 対話エンジン（Gemini一本＝slots更新＋提案）

1. 各ターンで **対話入力** か **条件選択** を選ぶ
2. 対話: Gemini がマージ済みslots＋聞き返し/案3つを JSON で返す
3. 条件選択: 固定リスト＋その他で複数スロットを直接更新 → 揃っていれば案3つ
4. Python 側で slots を保持。必須は `time_slot` / `area`
5. Gemini失敗時はエリア別ルールベース案

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
import re
import json

GEMINI_TIMEOUT_SEC = 60  # 抽出+提案を1回でやるので少し長め
SLOT_KEYS = ["budget", "time_slot", "area", "mood", "avoid_areas"]


def normalize_slots(raw_slots) -> dict:
    """Gemini JSON の slots を内部形式へ正規化。"""
    out = {}
    if not isinstance(raw_slots, dict):
        return out

    budget = raw_slots.get("budget")
    if budget is not None and budget != "":
        m = re.search(r"\d+", str(budget).replace(",", ""))
        if m:
            out["budget"] = int(m.group())

    for k in ["time_slot", "area", "mood"]:
        v = raw_slots.get(k)
        if v is None or v == "":
            continue
        s = str(v).strip()
        if s in ("空", "なし", "None", "null", "-", "未定"):
            continue
        out[k] = s

    avoid = raw_slots.get("avoid_areas")
    avoided = []
    if isinstance(avoid, list):
        avoided = [str(a).strip() for a in avoid if str(a).strip() and str(a).strip() not in ("空", "なし")]
    elif isinstance(avoid, str) and avoid.strip() and avoid.strip() not in ("空", "なし"):
        avoided = [p for p in re.split(r"[、,/]+", avoid) if p.strip()]
    if avoided:
        out["avoid_areas"] = list(dict.fromkeys(avoided))
        if out.get("area") in out["avoid_areas"]:
            out.pop("area", None)
    return out


def missing_slots(slots: dict):
    need = []
    if "time_slot" not in slots:
        need.append("時間帯（昼/午後/夜/終日など）")
    if "area" not in slots:
        need.append("エリア（渋谷・上野・新宿／お台場〜品川など）")
    return need


def template_clarify(need, slots):
    msg = "おけ。もうちょい条件ほしい。"
    if need:
        msg += " " + " / ".join(need) + " を教えて。"
    avoid = slots.get("avoid_areas")
    if avoid:
        msg += f" （除外中: {'、'.join(avoid)}）"
    msg += " 例:『上野で午後、一人5000円』"
    return msg


def fallback_plans(slots: dict) -> str:
    area = slots.get("area", "渋谷")
    budget = slots.get("budget", DEFAULT_BUDGET)
    time_slot = slots.get("time_slot", "昼/午後")
    mood = slots.get("mood", "コスパ重視")
    templates = {
        "上野": [
            (f"上野動物園 → 上野公園短散策 → 安めご飯（一人{budget}円以内）", "定番・コスパ"),
            (f"博物館（常設寄り）→ カフェ", "屋内・午後向き"),
            (f"公園散歩 → ラーメン/定食 → 早め解散", "ダラダラ回避"),
        ],
        "渋谷": [
            (f"公園短散策 → カフェ → 安めご飯", "集合しやすい"),
            (f"映画 → 前後でファミレス/ラーメン", "予算管理しやすい"),
            (f"スイーツカフェ → 短散歩", "早め解散可"),
        ],
        "新宿": [
            (f"新宿御苑 → ランチ", "緑＋コスパ"),
            (f"映画 → 安めご飯", "屋内中心"),
            (f"カフェ → 短散策", "落ち着き"),
        ],
        "お台場": [
            (f"海浜公園散歩 → 食事", "景色＋安さ"),
            (f"無料スポット中心 → チェーン食事", "高額回避"),
            (f"午後のんびり → 早め切り上げ", "終了明確"),
        ],
        "品川": [
            (f"水族館（料金要確認）→ 抑えめ食事", "予算調整"),
            (f"短散歩 → 安めご飯", "移動少なめ"),
            (f"屋内 → カフェ", "天候に強い"),
        ],
    }
    key = area if area in templates else "渋谷"
    lines = [f"（簡易提案 / {area} / {time_slot} / {mood} / {budget}円）", ""]
    for i, (plan, reason) in enumerate(templates[key], 1):
        lines += [f"案{i}: {plan}", f"理由: {reason}", ""]
    return "\n".join(lines).strip()


def prefs_for_gemini() -> str:
    return (
        "呼称: リョウスケ君/こゆたん。東京23区。"
        "優先エリア: 渋谷・上野・新宿/お台場〜品川。"
        "好き: 安いおいしいご飯、カフェ、映画、動物園/水族館、季節イベント、散歩、スイーツ。"
        "NG: 高いのに微妙、長時間ダラダラ。"
        "定番: イベント+ご飯、上野/渋谷起点、コスパ。"
    )


def _gemini_call(prompt: str) -> str:
    return gemini_rest_generate(prompt, GEMINI_MODEL_NAME, timeout=GEMINI_TIMEOUT_SEC)


def _extract_json_object(text: str) -> dict:
    """Gemini応答から最初のJSONオブジェクトを取り出す。"""
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t)
        t = re.sub(r"\s*```$", "", t)
    try:
        return json.loads(t)
    except Exception:
        pass
    m = re.search(r"\{[\s\S]*\}", t)
    if not m:
        raise ValueError("JSONオブジェクトが見つからない")
    return json.loads(m.group(0))


def format_plans_from_list(plans) -> str:
    """plans が list[{plan, reason}] のとき表示用文字列へ。"""
    if isinstance(plans, str) and plans.strip():
        return plans.strip()
    if not isinstance(plans, list):
        return ""
    lines = []
    for i, item in enumerate(plans[:3], 1):
        if isinstance(item, dict):
            plan = str(item.get("plan") or item.get("案") or "").strip()
            reason = str(item.get("reason") or item.get("理由") or "").strip()
        else:
            plan = str(item).strip()
            reason = ""
        if plan:
            lines.append(f"案{i}: {plan}")
        if reason:
            lines.append(f"理由: {reason}")
        if plan or reason:
            lines.append("")
    return "\n".join(lines).strip()


def gemini_turn(user_input: str, slots: dict) -> dict:
    """1回のGemini呼び出しで slotsマージ＋聞き返し or 案3つ。

    戻り値:
      {
        "slots": dict,
        "need_clarify": bool,
        "clarify_message": str|None,
        "plans": str|None,
        "raw": str,
      }
    """
    if not GEMINI_MODEL_NAME:
        merged = dict(slots)
        need = missing_slots(merged)
        if need:
            return {
                "slots": merged,
                "need_clarify": True,
                "clarify_message": template_clarify(need, merged),
                "plans": None,
                "raw": "",
            }
        merged.setdefault("budget", DEFAULT_BUDGET)
        return {
            "slots": merged,
            "need_clarify": False,
            "clarify_message": None,
            "plans": fallback_plans(merged),
            "raw": "",
        }

    prev = {
        "budget": slots.get("budget"),
        "time_slot": slots.get("time_slot"),
        "area": slots.get("area"),
        "mood": slots.get("mood"),
        "avoid_areas": slots.get("avoid_areas", []),
    }
    prompt = (
        "あなたはデート条件の抽出＋プラン提案アシスタント。\n"
        "予約代行・在庫確認・恋愛深掘り禁止。個人情報禁止。\n"
        "日本語で、次のJSONだけを返す（前後の説明・コードフェンス禁止）。\n"
        "{\n"
        '  "slots": {\n'
        '    "budget": <円の整数 or null>,\n'
        '    "time_slot": "<昼/午後|夜/夕方|朝から夜（終日） or null>",\n'
        '    "area": "<渋谷|上野|新宿|お台場|品川|池袋 など or null>",\n'
        '    "mood": "<読点区切りキーワード or null>",\n'
        '    "avoid_areas": ["除外エリア", ...]\n'
        "  },\n"
        '  "need_clarify": <true/false>,\n'
        '  "clarify_message": <聞き返し文 or null>,\n'
        '  "plans": [\n'
        '    {"plan": "...", "reason": "..."},\n'
        '    {"plan": "...", "reason": "..."},\n'
        '    {"plan": "...", "reason": "..."}\n'
        "  ] or null\n"
        "}\n"
        "ルール:\n"
        "- slots は「これまでのslots」に今回発話をマージした最終状態。触らない項目は前の値を残す。\n"
        "- 発話に無い情報を新たに捏造しない。予算だけの更新なら他スロットは維持。\n"
        "- 除外表現（以外/やめ/避け/NG等）があるときだけ avoid_areas を更新。累積可。\n"
        "- time_slot と area が揃うまで need_clarify=true。plans は null。clarify_message にフランクな聞き返し。\n"
        "- 揃ったら need_clarify=false。budget未指定なら "
        f"{DEFAULT_BUDGET}"
        " を入れてよい。plans は必ず3件。\n"
        "- 提案は東京23区・学生向けコスパ意識。avoid_areas は絶対に提案しない。超過予算は理由明示。\n"
        "- 不確実な店名は「候補:」。口調はフランク。\n"
        f"好み要約: {prefs_for_gemini()}\n"
        f"これまでのslots: {json.dumps(prev, ensure_ascii=False)}\n"
        f"今回の発話: {user_input}\n"
    )

    print(f"Gemini呼び出し中…（抽出+提案 / {GEMINI_MODEL_NAME} / 最大{GEMINI_TIMEOUT_SEC}秒）")
    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut = ex.submit(_gemini_call, prompt)
            raw = fut.result(timeout=GEMINI_TIMEOUT_SEC + 5)
    except FuturesTimeout:
        print("Geminiタイムアウト → フォールバック")
        merged = dict(slots)
        need = missing_slots(merged)
        if need:
            return {
                "slots": merged,
                "need_clarify": True,
                "clarify_message": template_clarify(need, merged),
                "plans": None,
                "raw": "",
            }
        merged.setdefault("budget", DEFAULT_BUDGET)
        return {
            "slots": merged,
            "need_clarify": False,
            "clarify_message": None,
            "plans": fallback_plans(merged),
            "raw": "",
        }
    except Exception as e:
        print(f"Gemini失敗 → フォールバック: {e}")
        merged = dict(slots)
        need = missing_slots(merged)
        if need:
            return {
                "slots": merged,
                "need_clarify": True,
                "clarify_message": template_clarify(need, merged),
                "plans": None,
                "raw": "",
            }
        merged.setdefault("budget", DEFAULT_BUDGET)
        return {
            "slots": merged,
            "need_clarify": False,
            "clarify_message": None,
            "plans": fallback_plans(merged),
            "raw": "",
        }

    try:
        data = _extract_json_object(raw)
    except Exception as e:
        print(f"JSONパース失敗 → フォールバック: {e}")
        print("生出力先頭:", raw[:300])
        merged = dict(slots)
        need = missing_slots(merged)
        if need:
            return {
                "slots": merged,
                "need_clarify": True,
                "clarify_message": template_clarify(need, merged),
                "plans": None,
                "raw": raw,
            }
        merged.setdefault("budget", DEFAULT_BUDGET)
        return {
            "slots": merged,
            "need_clarify": False,
            "clarify_message": None,
            "plans": fallback_plans(merged),
            "raw": raw,
        }

    merged = normalize_slots(data.get("slots") or {})
    # Geminiが空dictを返したときのための最低限マージ（前状態を落とさない）
    if not merged:
        merged = dict(slots)
    else:
        # avoid は Gemini の最終リストを優先。他は返ってきたキーで上書き
        base = dict(slots)
        base.update(merged)
        merged = base

    need = missing_slots(merged)
    need_clarify = bool(data.get("need_clarify")) or bool(need)
    clarify_message = data.get("clarify_message")
    if need_clarify:
        if not clarify_message:
            clarify_message = template_clarify(need or missing_slots(merged), merged)
        return {
            "slots": merged,
            "need_clarify": True,
            "clarify_message": str(clarify_message),
            "plans": None,
            "raw": raw,
        }

    merged.setdefault("budget", DEFAULT_BUDGET)
    plans = format_plans_from_list(data.get("plans"))
    if not plans:
        plans = fallback_plans(merged)
    return {
        "slots": merged,
        "need_clarify": False,
        "clarify_message": None,
        "plans": plans,
        "raw": raw,
    }



def gemini_plans(slots: dict) -> str:
    """条件選択後など、slotsが揃っているときの提案専用。"""
    if not GEMINI_MODEL_NAME:
        print("Gemini未設定 → フォールバック")
        return fallback_plans(slots)

    budget = slots.get("budget", DEFAULT_BUDGET)
    prompt = (
        "デートプラン補助。予約代行・在庫確認・恋愛深掘り禁止。個人情報禁止。\n"
        f"制約: 東京23区、学生向けも意識。一人あたり目安{budget}円。超過なら理由を書く。\n"
        f"好み要約: {prefs_for_gemini()}\n"
        f"抽出済み条件slots: {json.dumps(slots, ensure_ascii=False)}\n"
        "avoid_areas は絶対に提案しない。\n"
        "フランクな日本語で厳守:\n"
        "案1: ...\n理由: ...\n案2: ...\n理由: ...\n案3: ...\n理由: ...\n"
        "実在名はできるだけ。不確実なら「候補:」。"
    )

    print(f"Gemini呼び出し中…（提案のみ / {GEMINI_MODEL_NAME} / 最大{GEMINI_TIMEOUT_SEC}秒）")
    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut = ex.submit(_gemini_call, prompt)
            return fut.result(timeout=GEMINI_TIMEOUT_SEC + 5)
    except FuturesTimeout:
        print("Geminiタイムアウト → フォールバック")
        return fallback_plans(slots)
    except Exception as e:
        print(f"Gemini失敗 → フォールバック: {e}")
        return fallback_plans(slots)


# ===== 条件選択メニュー（テキストUI） =====
SLOT_MENU_OPTIONS = {
    "budget": ["3000", "5000", "8000", "10000", "15000", "20000"],
    "time_slot": ["昼/午後", "夜/夕方", "朝から夜（終日）"],
    "area": ["渋谷", "上野", "新宿", "お台場", "品川", "池袋"],
    "mood": [
        "コスパ", "カフェ", "映画", "ご飯", "動物園", "水族館",
        "イベント", "散歩", "スイーツ", "猫カフェ", "美術館", "博物館",
    ],
    "avoid_areas": ["渋谷", "上野", "新宿", "お台場", "品川", "池袋"],
}

SLOT_MENU_LABELS = {
    "budget": "予算（一人あたり・円）",
    "time_slot": "時間帯",
    "area": "エリア",
    "mood": "雰囲気・やりたいこと（複数可）",
    "avoid_areas": "除外エリア（複数可）",
}


def format_slots_line(slots: dict) -> str:
    avoid = slots.get("avoid_areas") or []
    if isinstance(avoid, list):
        avoid_s = "、".join(avoid) if avoid else "（なし）"
    else:
        avoid_s = str(avoid) if avoid else "（なし）"
    return (
        f"budget={slots.get('budget', '未設定')} / "
        f"time_slot={slots.get('time_slot', '未設定')} / "
        f"area={slots.get('area', '未設定')} / "
        f"mood={slots.get('mood', '未設定')} / "
        f"avoid_areas={avoid_s}"
    )


def _parse_index_list(raw: str, n: int):
    """'1,3,5' -> 0-based indices. 空は []."""
    raw = (raw or "").strip()
    if not raw:
        return []
    out = []
    for part in re.split(r"[、,\s]+", raw):
        if not part:
            continue
        if not part.isdigit():
            raise ValueError(f"番号以外: {part}")
        i = int(part)
        if i < 1 or i > n:
            raise ValueError(f"範囲外: {i}")
        if (i - 1) not in out:
            out.append(i - 1)
    return out


def _ask_single_choice(label: str, options: list, current):
    """単一選択。戻り値: 新しい値 or None（クリア） or Ellipsis（変更なし）。"""
    print(f"\n[{label}] 現在: {current if current not in (None, '') else '未設定'}")
    for i, opt in enumerate(options, 1):
        print(f"  {i}. {opt}")
    print(f"  {len(options)+1}. その他（自由入力）")
    print("  0. クリア（未設定）")
    print("  Enter. 変更しない")
    raw = input("番号: ").strip()
    if raw == "":
        return Ellipsis
    if raw == "0":
        return None
    if raw.isdigit() and 1 <= int(raw) <= len(options):
        return options[int(raw) - 1]
    if raw.isdigit() and int(raw) == len(options) + 1:
        other = input("自由入力: ").strip()
        return other if other else Ellipsis
    # 数字以外はそのまま自由入力として受け付ける
    return raw


def _ask_multi_choice(label: str, options: list, current):
    """複数選択。戻り値: list / None(クリア) / Ellipsis(変更なし) / str(結合済みも可→list化)。"""
    if isinstance(current, list):
        cur_s = "、".join(current) if current else "未設定"
    else:
        cur_s = current if current not in (None, "") else "未設定"
    print(f"\n[{label}] 現在: {cur_s}")
    for i, opt in enumerate(options, 1):
        print(f"  {i}. {opt}")
    print(f"  {len(options)+1}. その他（自由入力を追加）")
    print("  0. クリア（未設定）")
    print("  Enter. 変更しない")
    print("  例: 1,3  /  2,その他は続けて入力")
    raw = input("番号（カンマ区切り可）: ").strip()
    if raw == "":
        return Ellipsis
    if raw == "0":
        return None

    chosen = []
    want_other = False
    for part in re.split(r"[、,]+", raw):
        part = part.strip()
        if not part:
            continue
        if part.isdigit():
            i = int(part)
            if i == 0:
                return None
            if 1 <= i <= len(options):
                v = options[i - 1]
                if v not in chosen:
                    chosen.append(v)
            elif i == len(options) + 1:
                want_other = True
            else:
                print(f"  無視した番号: {i}")
        else:
            if part not in chosen:
                chosen.append(part)
    if want_other:
        other = input("その他（読点区切り可）: ").strip()
        if other:
            for p in re.split(r"[、,/]+", other):
                p = p.strip()
                if p and p not in chosen:
                    chosen.append(p)
    return chosen if chosen else Ellipsis


def edit_slots_menu(slots: dict) -> dict:
    """複数スロットをまとめて編集して返す（新dict）。"""
    slots = dict(slots)
    print("\n======= 条件選択 =======")
    print("現在:", format_slots_line(slots))
    keys = ["budget", "time_slot", "area", "mood", "avoid_areas"]
    print("変えたい項目を番号で選んで（複数可）。例: 1,3,4")
    for i, k in enumerate(keys, 1):
        print(f"  {i}. {SLOT_MENU_LABELS[k]} ({k})")
    print("  Enter. 何も変えず戻る")
    raw = input("項目番号: ").strip()
    if raw == "":
        print("変更なし。")
        return slots
    try:
        idxs = _parse_index_list(raw, len(keys))
    except ValueError as e:
        print("入力エラー:", e)
        return slots
    if not idxs:
        return slots

    for idx in idxs:
        key = keys[idx]
        opts = SLOT_MENU_OPTIONS[key]
        if key in ("mood", "avoid_areas"):
            cur = slots.get(key)
            if key == "mood" and isinstance(cur, str):
                cur = [p for p in re.split(r"[、,/]+", cur) if p]
            val = _ask_multi_choice(SLOT_MENU_LABELS[key], opts, cur)
            if val is Ellipsis:
                continue
            if val is None or val == []:
                slots.pop(key, None)
            elif key == "mood":
                slots["mood"] = "、".join(val)
            else:
                slots["avoid_areas"] = list(val)
        else:
            cur = slots.get(key)
            val = _ask_single_choice(SLOT_MENU_LABELS[key], opts, cur)
            if val is Ellipsis:
                continue
            if val is None or val == "":
                slots.pop(key, None)
            elif key == "budget":
                m = re.search(r"\d+", str(val).replace(",", ""))
                if m:
                    slots["budget"] = int(m.group())
                else:
                    print("予算が数字として読めなかったのでスキップ")
            else:
                slots[key] = str(val)

    # 希望エリアが除外に入っていたら希望を消す
    avoided = slots.get("avoid_areas") or []
    if slots.get("area") in avoided:
        print("注意: area が avoid_areas に含まれるため area をクリアしたよ。")
        slots.pop("area", None)

    print("更新後:", format_slots_line(slots))
    return slots

print("engine ready (Gemini一本 = slots更新 + 案3つ)")

## 8. 対話テスト（exit で終了）

各ターン開始時に `1: 対話入力` / `2: 条件選択` を選ぶ。`reset` で条件クリア、`exit` で終了。

In [ ]:
slots = {}
print("デートBot起動（Gemini一本モード）")
print("各ターン最初に 1=対話入力 / 2=条件選択。 reset / exit も可")
print("例（対話）: 『こゆたんと土曜の午後、上野で安く遊びたい』")

while True:
    print("\n----- ターン開始 -----")
    print("現在の条件:", format_slots_line(slots))
    mode = input("1: 対話入力 / 2: 条件選択 / reset / exit > ").strip().lower()
    if not mode:
        continue
    if mode in ("exit", "q", "quit"):
        break
    if mode == "reset":
        slots = {}
        print("Bot: 条件リセットしたよ。")
        continue
    if mode not in ("1", "2", "対話", "対話入力", "条件", "条件選択"):
        print("1 か 2（または reset / exit）を選んでね。")
        continue

    # --- 条件選択 ---
    if mode in ("2", "条件", "条件選択"):
        slots = edit_slots_menu(slots)
        need = missing_slots(slots)
        if need:
            print("Bot:", template_clarify(need, slots))
            continue
        slots.setdefault("budget", DEFAULT_BUDGET)
        plans = gemini_plans(slots)
        print("Bot（提案）:\n" + plans)
        print("(現在の条件)", slots)
        continue

    # --- 対話入力 ---
    user_input = input("あなた: ").strip()
    if not user_input:
        continue
    if user_input.lower() == "exit":
        break
    if user_input.lower() == "reset":
        slots = {}
        print("Bot: 条件リセットしたよ。")
        continue

    if any(k in user_input for k in ["予約して", "空席", "恋愛相談", "復縁", "住所", "電話番号"]):
        print("Bot: それは対応外だよ。デート案の相談だけ続けるね。")
        continue

    if len(user_input) <= 1:
        print("Bot:", template_clarify(["時間帯", "エリア"], slots))
        print("(現在の条件)", slots)
        continue

    result = gemini_turn(user_input, slots)
    slots = result["slots"]
    print("(マージ後slots)", slots)

    if result["need_clarify"]:
        print("Bot:", result["clarify_message"])
        continue

    print("Bot（提案）:\n" + (result["plans"] or ""))
    print("(現在の条件)", slots)

## 9. 運用メモ（Gemini一本モード）

1. Secrets の `GEMINI_API_KEY` があれば、セクション0 → 1 → 6 → 7 → 8 の順で実行
2. セクション2〜5はスキップ（実行不要）
3. 無料枠のレート制限に当たったら少し待って再実行
4. LoRA版に戻すときは旧ノート＋ Drive の `date_bot_lora`（`docs/SAVE_LOAD.md`）

GitHub公開時は APIキー・LINE生ログ・本番CSVは載せない。
